In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc ffsim matplotlib numpy pyscf qiskit-addon-sqd
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Квантова діагоналізація на основі вибірок хімічного гамільтоніана

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Оцінка використання: менше однієї хвилини на процесорі Heron r2 (ПРИМІТКА: Це лише оцінка. Твій час виконання може відрізнятися.)*

## Результати навчання
Після проходження цього посібника користувачі повинні розуміти:
- Як використовувати [SQD Qiskit addon](https://github.com/Qiskit/qiskit-addon-sqd) для апроксимації енергії основного стану молекулярної системи за допомогою бітових рядків, відібраних з квантового процесора (QPU).
- Як використовувати [ffsim](https://github.com/qiskit-community/ffsim) для побудови схеми локального унітарного кластерного Яструва (LUCJ) для квантового хімічного моделювання.

## Передумови
Рекомендуємо користувачам ознайомитися з такими темами перед початком роботи з цим посібником:
- Квантова хімія та друге квантування
- Використання примітива Sampler для відбору вибірок з квантових схем

## Передісторія
У цьому посібнику ми показуємо, як виконати постобробку шумних квантових вибірок для апроксимації основного стану молекули азоту $\text{N}_2$ при рівноважній довжині зв'язку, використовуючи [SQD Qiskit addon](https://github.com/Qiskit/qiskit-addon-sqd) для реалізації [алгоритму квантової діагоналізації на основі вибірок (SQD)](https://arxiv.org/abs/2405.05068). Докладніше про програмне забезпечення можна дізнатися у відповідній [документації](/guides/qiskit-addons-sqd), включно з [простим прикладом](/guides/qiskit-addons-sqd-get-started) для початку роботи.

Цей посібник рекомендований користувачам, які знайомі з квантовою хімією: зокрема, з пошуком енергій основного стану молекули. Для детального ознайомлення з робочим процесом звернися до [курсу з алгоритмів квантової діагоналізації](https://quantum.cloud.ibm.com/learning/courses/quantum-diagonalization-algorithms).

SQD — це техніка пошуку власних значень і власних векторів квантових операторів, таких як гамільтоніан квантової системи, за допомогою спільного використання квантових і розподілених класичних обчислень. Розподілені класичні обчислення використовуються для обробки вибірок, отриманих з квантового процесора, та для проектування й діагоналізації цільового гамільтоніана у підпросторі, що ними охоплюється. Робочий процес на основі SQD має такі кроки:

1.  Обери схему-анзац і застосуй її на квантовому комп'ютері до опорного стану (у цьому випадку — стан [Хартрі-Фока](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method)).
2.  Відбери бітові рядки з отриманого квантового стану.
3.  Запусти процедуру *самоузгодженого відновлення конфігурацій* над бітовими рядками для отримання апроксимації основного стану.

SQD добре працює тоді, коли цільовий власний стан є розрідженим: хвильова функція підтримується на наборі базисних станів $\mathcal{S} = {|x\rangle }$, розмір якого не зростає експоненційно зі збільшенням розміру задачі.

### Квантова хімія
Гамільтоніан молекулярної системи можна записати як

$$
\hat{H} = \sum_{ \substack{pr\\\sigma} } h_{pr} \, \hat{a}^\dagger_{p\sigma} \hat{a}_{r\sigma}
+ \frac12
\sum_{ \substack{prqs\\\sigma\tau} }
h_{prqs} \,
\hat{a}^\dagger_{p\sigma}
\hat{a}^\dagger_{q\tau}
\hat{a}_{s\tau}
\hat{a}_{r\sigma},
$$

де $h_{pr}$ і $h_{prqs}$ — комплексні числа, що називаються молекулярними інтегралами і можуть бути обчислені зі специфікації молекули за допомогою комп'ютерної програми. У цьому посібнику ми обчислюємо інтеграли за допомогою програмного пакета [PySCF](https://pyscf.org/).

Щоб дізнатися більше про те, як виводиться молекулярний гамільтоніан, звернися до підручника з квантової хімії (наприклад, *Modern Quantum Chemistry* Сзабо і Остлунда). Для загального пояснення того, як задачі квантової хімії відображаються на квантові комп'ютери, переглянь лекцію [*Mapping Problems to Qubits*](https://youtube.com/watch?v=TyFU6r8uEsE\&t=900) з Qiskit Global Summer School 2024.

### Анзац локального унітарного кластерного Яструва (LUCJ)
SQD потребує квантову схему-анзац для відбору вибірок. У цьому посібнику ми використаємо анзац [локального унітарного кластерного Яструва (LUCJ)](https://pubs.rsc.org/en/content/articlelanding/2023/sc/d3sc02516k) завдяки поєднанню фізичної обґрунтованості та апаратної дружності. Для побудови схеми анзацу ми використаємо [ffsim](https://qiskit-community.github.io/ffsim/).

Анзац LUCJ адаптується до QPU з обмеженою зв'язністю кубітів. Спінові орбіталі відображаються на кубіти таким чином, що анзац не потребує маршрутизації зі SWAP-воротами. Обладнання IBM&reg; має топологію кубітів типу важкого гексагону, і в цьому випадку ми можемо використати зиґзаґоподібний шаблон, зображений нижче. У цьому шаблоні орбіталі з однаковим спіном відображаються на кубіти з лінійною топологією (червоні та сині кола), а зв'язок між орбіталями різного спіну присутній кожні 4 просторові орбіталі, де зв'язок забезпечується допоміжним кубітом (фіолетові кола).

![Діаграма відображення кубітів для анзацу LUCJ на решітці важкого гексагону](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/7e0ee7e1-2d24-417f-ac59-25c58db79aa9.avif)

### Самоузгоджене відновлення конфігурацій
Процедура самоузгодженого відновлення конфігурацій призначена для вилучення якомога більшого сигналу з шумних квантових вибірок. Оскільки молекулярний гамільтоніан зберігає число частинок і спін Z, доцільно обрати схему-анзац, яка також зберігає ці симетрії. При застосуванні до стану Хартрі-Фока, отриманий стан матиме фіксоване число частинок і спін Z у безшумних умовах. Тому половини зі спіном-$\alpha$ і спіном-$\beta$ будь-якого бітового рядка, відібраного з цього стану, повинні мати таку саму [вагу Геммінга](https://en.wikipedia.org/wiki/Hamming_weight), як і в стані Хартрі-Фока. Через присутність шуму в сучасних квантових процесорах деякі виміряні бітові рядки порушуватимуть цю властивість. Проста форма постселекції відкидала б такі бітові рядки, але це марнотратно, адже вони можуть ще містити певний сигнал. Процедура самоузгодженого відновлення намагається відновити частину цього сигналу при постобробці. Процедура є ітеративною і потребує на вході оцінку середньої заповненості кожної орбіталі в основному стані, яка спочатку обчислюється з необроблених вибірок. Процедура виконується в циклі, і кожна ітерація має такі кроки:

1.  Для кожного бітового рядка, що порушує задані симетрії, перевертай його біти за допомогою імовірнісної процедури, призначеної для наближення бітового рядка до поточної оцінки середньої орбітальної заповненості, щоб отримати новий бітовий рядок.
2.  Зібери всі старі та нові бітові рядки, що відповідають симетріям, і відбери підмножини фіксованого розміру, обраного заздалегідь.
3.  Для кожної підмножини бітових рядків проектуй гамільтоніан у підпростір, що охоплюється відповідними базисними векторами (опис цих базисних векторів дивись у [попередньому розділі](#quantum-chemistry)), і обчисли оцінку основного стану спроектованого гамільтоніана на класичному комп'ютері.
4.  Оновлюй оцінку середньої орбітальної заповненості за допомогою оцінки основного стану з найнижчою енергією.

### Діаграма робочого процесу SQD
Робочий процес SQD зображено на такій діаграмі:

![Діаграма робочого процесу алгоритму SQD](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/fd7e816f-4e2e-4dd7-a7da-f71afb9ca68d.avif)

## Вимоги
Перед початком цього посібника переконайся, що у тебе встановлено таке:

*   Qiskit SDK v1.0 або новіший, з підтримкою [візуалізації](https://docs.quantum.ibm.com/api/qiskit/visualization)
*   Qiskit Runtime v0.22 або новіший (`pip install qiskit-ibm-runtime`)
*   SQD Qiskit addon v0.11 або новіший (`pip install qiskit-addon-sqd`)
*   ffsim v0.0.75 або новіший (`pip install ffsim`)

## Налаштування

In [1]:
import math

import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Маломасштабний приклад на симуляторі

У цьому посібнику ми знайдемо апроксимацію основного стану молекули азоту поблизу її рівноважної відстані зв'язку. Спочатку ми використаємо малий базисний набір STO-6G, щоб змоделювати експеримент і переконатися, що він працює.

### Крок 1: Відображення класичних вхідних даних на квантову задачу

Спочатку вкажемо молекулу та її властивості.

In [2]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="sto-6g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -108.464957764796
CASCI E = -108.595987350986  E(CI) = -32.4115475088426  S^2 = 0.0000000
norb = 8
nelec = (5, 5)


Before constructing the LUCJ ansatz circuit, we first perform a CCSD calculation in the following code cell. The [$t_1$ and $t_2$ amplitudes](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) from this calculation will be used to initialize the parameters of the ansatz.

In [3]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -108.5933309085008  E_corr = -0.1283731437052354


Перед побудовою схеми анзацу LUCJ ми спочатку виконуємо розрахунок CCSD у такій комірці коду. Амплітуди [$t_1$ і $t_2$](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) з цього розрахунку будуть використані для ініціалізації параметрів анзацу.

In [ ]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]

# Let generate_lucj_pass_manager determine the alpha-beta interactions
pairs_ab = None

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell
    # from running too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it
# to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

### Step 2: Optimize for quantum hardware execution

Next, we optimize the circuit for a target hardware. Typically, this step involves initializing the hardware backend and a pass manager for that backend. However, since the LUCJ ansatz is adapted to the hardware connectivity, we already performed these actions in the previous step. All that is left to do is run the pass manager on the circuit to transpile it to an ISA circuit that can be directly executed on the QPU.

In [5]:
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")

Gate counts: OrderedDict({'xx_plus_yy': 86, 'p': 16, 'measure': 16, 'cp': 15, 'x': 10, 'swap': 2, 'barrier': 1})


Тепер ми використовуємо [ffsim](https://github.com/qiskit-community/ffsim) для створення схеми анзацу. Оскільки наша молекула має замкнутий стан Хартрі-Фока, ми використовуємо спін-збалансований варіант анзацу UCJ, [UCJOpSpinBalanced](https://qiskit-community.github.io/ffsim/api/ffsim.html#ffsim.UCJOpSpinBalanced). Ми встановлюємо `optimize=True` у методі `from_t_amplitudes`, щоб увімкнути «стиснуте» подвійне факторизування амплітуд $t_2$ (дивись [The local unitary cluster Jastrow (LUCJ) ansatz](https://qiskit-community.github.io/ffsim/explanations/lucj.html#Parameter-initialization-from-CCSD) у документації ffsim для деталей).

Оскільки анзац LUCJ адаптується до доступної зв'язності QPU, ми маємо ініціалізувати бекенд QPU перед створенням анзацу. Зараз ми створимо загальний бекенд із картою зв'язності важкого гексагону і набором воріт, до якого анзац LUCJ природно розкладається. Потім ми використаємо `ffsim.qiskit.generate_lucj_pass_manager` для створення менеджера проходів, що спеціалізується на транспіляції анзацу LUCJ до заданого бекенду згідно з зиґзаґоподібним компонуванням, описаним у [розділі про анзац LUCJ](#local-unitary-cluster-jastrow-lucj-ansatz). Ця функція використовує евристику оцінки для мінімізації помилок, пов'язаних з обраним компонуванням, що важливо, якщо твій бекенд є реальним QPU або симулятором з моделлю шуму. Крім повернення менеджера проходів, ця функція також повертає пари зв'язку альфа-бета, які можна реалізувати на обладнанні. Якщо реалізувати не всі пари, вона видасть попередження.

In [6]:
rng = np.random.default_rng()
sampler = StatevectorSampler(seed=rng)
job = sampler.run([isa_circuit], shots=100_000)

In [7]:
primitive_result = job.result()
pub_result = primitive_result[0]

### Step 4: Post-process and return result in desired classical format

A useful metric to judge the quality of the QPU output is the number of valid configurations returned. A valid configuration has the correct particle number and spin Z, which means that the right half of the bitstring has Hamming weight equal to the number of spin-up electrons, and the left half has Hamming weight equal to the number of spin-down electrons. The following cell computes the fraction of sampled configurations that are valid.

In [8]:
def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )


bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

Fraction of sampled configurations that are valid: 1.0


### Крок 3: Виконання за допомогою примітивів Qiskit
Після оптимізації схеми для виконання на обладнанні ми готові запустити її на цільовому обладнанні та зібрати вибірки для оцінки енергії основного стану. Оскільки у нас лише одна схема, ми будемо використовувати [режим виконання Job](/guides/execution-modes) Qiskit Runtime і виконаємо нашу схему.

In [ ]:
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: "
    f"{expected_fraction_random}"
)

Expected fraction of valid configurations from uniformly random bitstrings: 0.0478515625


Now, we estimate the ground state energy of the Hamiltonian using the `diagonalize_fermionic_hamiltonian` function. This function performs the self-consistent configuration recovery procedure to iteratively refine the noisy quantum samples to improve the energy estimate. We pass a callback function so that we can save the intermediate results for later analysis. See the [API documentation](/docs/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian) for explanations of the arguments to `diagonalize_fermionic_hamiltonian`.

Here, we use the `initial_occupancies` argument to `diagonalize_fermionic_hamiltonian` to specify the Hartree-Fock configuration as the initial guess for the orbital occupancies in the ground state. This approach is sensible for systems where the ground state has significant support on the Hartree-Fock configuration, but it might not be appropriate in other situations, though more advanced computational methods might yield better initial guesses in those cases. Specifying `initial_occupancies` also allows configuration recovery to run even if no valid configurations were sampled, as may be the case when sampling a large circuit on a noisy QPU. Without this argument, the configuration recovery would fail and raise an error if no valid configurations were provided.

In [10]:
from functools import partial

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

Iteration 1
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Iteration 2
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Final energy: -108.59275573641656
Final energy error: 0.0032316145694579745


#### Visualize the results

The first plot shows that in this simulation we are already within `1 mH` of the exact answer after the first iteration (chemical accuracy is typically accepted to be `1 kcal/mol` $\approx$ `1.6 mH`). This is a small system, though, and because the samples are noiseless, configuration recovery is not needed. On a larger system run on a noisy QPU, multiple configuration recovery iterations might be needed, and the final accuracy might be worse. Generally, the energy can be improved by allowing more configuration recovery iterations or by increasing the number of samples per batch.

The second plot shows the average occupancy of each spatial orbital after the final iteration. We can see that both the spin-up and spin-down electrons occupy the first five orbitals with high probability in our solutions.

In [11]:
# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/caffd888-e89c-4aa9-8bae-4d1bb723b35e-0.avif" alt="Output of the previous code cell" />

### Крок 4: Постобробка та повернення результату у бажаному класичному форматі
Корисною метрикою для оцінки якості виводу QPU є кількість повернутих дійсних конфігурацій. Дійсна конфігурація має правильне число частинок і спін Z, тобто права половина бітового рядка має вагу Геммінга, рівну кількості електронів зі спіном вгору, а ліва половина — рівну кількості електронів зі спіном вниз. У такій комірці обчислюється частка відібраних конфігурацій, що є дійсними.

In [ ]:
# ------------------------------ Step 1 ------------------------------
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="cc-pvdz",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Store reference energy from SCI calculation performed separately
reference_energy = -109.22802921665716

print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]

# Let generate_lucj_pass_manager determine the alpha-beta interactions
pairs_ab = None

# Initialize backend
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(f"Using backend {backend.name}")

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell
    # from running too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it
# to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()


# ------------------------------ Step 2 ------------------------------

isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")


# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_SQD"]
job = sampler.run([isa_circuit], shots=100_000)
primitive_result = job.result()
pub_result = primitive_result[0]


# ------------------------------ Step 4 ------------------------------

bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: "
    f"{expected_fraction_random}"
)
# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the
# orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the
# sci_solver argument in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

converged SCF energy = -108.929838385609
norb = 26
nelec = (5, 5)
E(CCSD) = -109.2177884185544  E_corr = -0.2879500329450045
Using backend ibm_boston


Removing interaction (24, 24) from the end.
Removing interaction (20, 20) from the end.


Gate counts: OrderedDict({'sx': 7039, 'rz': 6990, 'cz': 1858, 'x': 61, 'measure': 52, 'barrier': 1})
Fraction of sampled configurations that are valid: 0.02124
Expected fraction of valid configurations from uniformly random bitstrings: 9.607888706852918e-07
Iteration 1
	Subsample 0
		Energy: -109.13889134249762
		Subspace dimension: 120409
	Subsample 1
		Energy: -109.11785470455858
		Subspace dimension: 110889
	Subsample 2
		Energy: -109.13234360554011
		Subspace dimension: 130321
Iteration 2
	Subsample 0
		Energy: -109.16392179579177
		Subspace dimension: 223729
	Subsample 1
		Energy: -109.16281938332986
		Subspace dimension: 223729
	Subsample 2
		Energy: -109.16955816711932
		Subspace dimension: 233289
Iteration 3
	Subsample 0
		Energy: -109.17905772999075
		Subspace dimension: 324900
	Subsample 1
		Energy: -109.17532445048462
		Subspace dimension: 357604
	Subsample 2
		Energy: -109.1733168689756
		Subspace dimension: 348100
Iteration 4
	Subsample 0
		Energy: -109.18437778820451
		Su

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/3858949c-a55d-4ff8-a0fc-54fb53e131b5-3.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](/docs/tutorials/sample-based-krylov-quantum-diagonalization) - a related tutorial using time evolution circuits instead of a variational ansatz
- [Scale SQD chemistry workflows with Dice solver](/docs/addons/qiskit-addon-sqd/guides/integrate-dice-solver) - a page showing how to use the more efficient Dice software for diagonalization
- [SQD addon API documentation](/docs/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian) - reference for the `diagonalize_fermionic_hamiltonian` function
- [*Chemistry beyond the scale of exact diagonalization on a quantum-centric supercomputer*](https://www.science.org/doi/10.1126/sciadv.adu9991) - the paper this tutorial is based on
</Admonition>